## 0)  Project Root

In [2]:
import sys
from pathlib import Path


def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()

[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


## 2) Device Selection

In [3]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Evaluating on:", device)

Evaluating on: mps


## 3) Folders

In [31]:
from pathlib import Path
import json
import torch


SEQ_DIR = PROJECT_ROOT / "03_Sequences"
RUN_DIR = PROJECT_ROOT / "runs" / "LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC"  # <- pick one


CONFIG_PATH = RUN_DIR / "config.json"
CKPT_PATH   = RUN_DIR / "best_model.pt"


#########################################################

In [46]:
from utils.data_utils import make_test_loader
import hashlib

test_loader, X_test_raw, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

model_class_name = config["model_class"]
model_kwargs     = config["model_kwargs"]
seq_len = config["seq_len"]
SEQ_LEAF_DIR = SEQ_DIR / f"seq{seq_len}"   # SEQ_DIR is base: .../03_Sequences



# Check hashes of loaded test data
def md5_np(a): 
    return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()

print("RAW X md5:", md5_np(X_test_raw))   # or X_test if you loaded raw
print("RAW y md5:", md5_np(y_test))
print("RAW shape:", X_test_raw.shape, y_test.shape)

Xb, yb = next(iter(test_loader))
print("BATCH0 X md5 (new):", md5_np(Xb.detach().cpu().numpy()))
print("BATCH0 y md5 (new):", md5_np(yb.detach().cpu().numpy()))

RAW X md5: 821e37fd2e30d0dd30749865b5d2ab94
RAW y md5: 7138a0d4459944e41259e786b11688ec
RAW shape: (473458, 60, 15) (473458,)
BATCH0 X md5 (new): 061202e8dccdec9976d5dab5c5c1cb22
BATCH0 y md5 (new): eb4c4dd77b8e0c0b5461a1769168aada


In [41]:
from models import LSTMClassifier  # add more as you create them
import hashlib

MODEL_REGISTRY = {
    "LSTMClassifier": LSTMClassifier,
    # "GRUClassifier": GRUClassifier,
    # "TransformerClassifier": TransformerClassifier,
}

ModelClass = MODEL_REGISTRY[model_class_name]
best_model = ModelClass(**model_kwargs).to(device)
best_model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
best_model.eval()

def fp(m):
    h = hashlib.sha256()
    for k in sorted(m.state_dict().keys()):
        t = m.state_dict()[k].detach().cpu().contiguous().numpy()
        h.update(k.encode()); h.update(t.tobytes())
    return h.hexdigest()[:16]

print("MODEL_FP:", fp(best_model))

print("Model loaded from:", CKPT_PATH)


MODEL_FP: 6eae0c985503c3f6
Model loaded from: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC/best_model.pt


In [42]:
import numpy as np
from tqdm import tqdm

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(tqdm(test_loader, desc="Predicting (test)", leave=False)):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)                 # ensure shape (batch,)
        probs  = torch.sigmoid(logits)                   # UP probability in [0,1]
        preds  = (probs >= 0.5).long()

        ########################################
        # DEBUG: first batch only
        if batch_i == 0:
            # 1) Model fingerprint (again, right now)
            import hashlib, torch
            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))

            # 2) Input fingerprint (whole tensor, not just 5 values)
            xb = X_batch.detach().cpu().contiguous().numpy()
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])

            # 3) Logits fingerprint
            lg = logits.detach().cpu().contiguous().numpy()
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])

            # 4) Preds fingerprint
            pr = preds.detach().cpu().contiguous().numpy()
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])
            
            print("model.training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)
        ########################################

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_probs  = np.array(all_probs).reshape(-1)
all_preds  = np.array(all_preds).reshape(-1).astype(int)
all_labels = np.array(all_labels).reshape(-1).astype(int)


Predicting (test):   0%|          | 35/29592 [00:00<02:50, 172.98it/s]

MODEL_FP: 6eae0c985503c3f6
X_FP: bafdebb3d1c993b8
LOGITS_FP: 8a4b4452a9b2f74a
PREDS_FP: 76c862828aa3b269
model.training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0


KeyboardInterrupt: 

In [8]:
meta = json.loads((SEQ_LEAF_DIR / "meta.json").read_text())

t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
t_to_end_min_values = X_test_raw[:, -1, t_to_end_min_idx].astype(int)

assert len(t_to_end_min_values) == len(all_labels) == len(all_probs)


In [9]:
import pandas as pd

df = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "p_up": all_probs,
    "y_pred": all_preds,
})


In [10]:
import numpy as np
import json

df["tp"] = ((df.y_true==1) & (df.y_pred==1)).astype(int)
df["fp"] = ((df.y_true==0) & (df.y_pred==1)).astype(int)
df["tn"] = ((df.y_true==0) & (df.y_pred==0)).astype(int)
df["fn"] = ((df.y_true==1) & (df.y_pred==0)).astype(int)
df["correct"] = (df.y_true == df.y_pred).astype(int)

stats = df.groupby("t_to_end_min").agg(
    count=("correct","count"),
    tp=("tp","sum"),
    fp=("fp","sum"),
    tn=("tn","sum"),
    fn=("fn","sum"),
    accuracy=("correct","mean"),
    avg_p_up=("p_up","mean"),
    base_rate=("y_true","mean"),
).reset_index()

# -----------------------------
# Add all classification metrics
# -----------------------------
def safe_div(num, den):
    return np.where(den == 0, np.nan, num / den)

# Positive-class metrics (UP = 1)
stats["precision"] = safe_div(stats["tp"], stats["tp"] + stats["fp"])   # PPV
stats["recall"]    = safe_div(stats["tp"], stats["tp"] + stats["fn"])   # TPR / Sensitivity

stats["f1"] = safe_div(
    2 * stats["precision"] * stats["recall"],
    stats["precision"] + stats["recall"]
)

# Negative-class metrics (DOWN = 0)
stats["specificity"] = safe_div(stats["tn"], stats["tn"] + stats["fp"]) # TNR
stats["npv"]         = safe_div(stats["tn"], stats["tn"] + stats["fn"]) # Negative Predictive Value

# Optional but common:
stats["fpr"] = safe_div(stats["fp"], stats["fp"] + stats["tn"])         # False Positive Rate
stats["fnr"] = safe_div(stats["fn"], stats["fn"] + stats["tp"])         # False Negative Rate

# Percent formats (optional)
stats["accuracy_pct"] = stats["accuracy"] * 100
stats["precision_pct"] = stats["precision"] * 100
stats["recall_pct"] = stats["recall"] * 100
stats["f1_pct"] = stats["f1"] * 100
stats["specificity_pct"] = stats["specificity"] * 100
stats["npv_pct"] = stats["npv"] * 100

# -----------------------------
# Save to JSON
# -----------------------------
stats_json = stats.to_dict(orient="records")

OUT_DIR = RUN_DIR / "eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "per_t_to_end_min_stats.json"
out_path.write_text(json.dumps(stats_json, indent=2))

print(stats.to_string(index=False))
print(f"\nSaved per-minute stats to {out_path}")


 t_to_end_min  count    tp   fp    tn   fn  accuracy  avg_p_up  base_rate  precision   recall       f1  specificity      npv      fpr      fnr  accuracy_pct  precision_pct  recall_pct    f1_pct  specificity_pct   npv_pct
            1  31564 15370  364 15459  371  0.976714  0.484477   0.498701   0.976865 0.976431 0.976648     0.976996 0.976563 0.023004 0.023569     97.671398      97.686539   97.643098 97.664813        97.699551 97.656349
            2  31564 14768 1073 14750  973  0.935179  0.485368   0.498701   0.932264 0.938187 0.935216     0.932187 0.938116 0.067813 0.061813     93.517932      93.226438   93.818690 93.521626        93.218732 93.811614
            3  31564 14360 1700 14123 1381  0.902389  0.489892   0.498701   0.894147 0.912267 0.903116     0.892561 0.910926 0.107439 0.087733     90.238880      89.414695   91.226733 90.311625        89.256146 91.092621
            4  31564 13963 2157 13666 1778  0.875333  0.493830   0.498701   0.866191 0.887047 0.876495     0.863679 

## Calibration & Threshold

In [11]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calibration_curve(df_sub, bins=10, min_count=5):
    df_sub = df_sub.copy()
    df_sub["prob_bin"] = pd.cut(df_sub["p_up"], bins=bins)

    calib = df_sub.groupby("prob_bin").agg(
        avg_p=("p_up", "mean"),
        freq_up=("y_true", "mean"),
        count=("y_true", "count"),
    ).dropna()

    return calib[calib["count"] >= min_count]


# ----------------------------
# Calibration plots per t
# ----------------------------
OUT_DIR = RUN_DIR / "eval" / "calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for t, g in df.groupby("t_to_end_min"):
    calib = calibration_curve(g, bins=10, min_count=5)
    if len(calib) < 2:
        continue

    total_n = len(g)

    plt.figure()

    # Model calibration curve
    plt.plot(
        calib["avg_p"],
        calib["freq_up"],
        marker="o",
        label="Model calibration"
    )

    # Perfect calibration reference
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="green",
        label="Perfect calibration"
    )

    # Annotate each point with absolute + percentage
    for _, row in calib.iterrows():
        pct = 100 * row["count"] / total_n
        plt.annotate(
            f"n={int(row['count'])}\n({pct:.1f}%)",
            (row["avg_p"], row["freq_up"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
        )

    # Axis ticks at 0.1
    ticks = np.linspace(0, 1, 11)
    plt.xticks(ticks)
    plt.yticks(ticks)
    plt.grid(True, which="both", linestyle="--", alpha=0.5)

    plt.xlabel("Average predicted probability")
    plt.ylabel("Empirical UP frequency")
    plt.title(f"Calibration curve (t_to_end_min={t}, n={total_n})")
    plt.legend(loc="lower right")

    plt.savefig(
        OUT_DIR / f"calibration_t{t}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()

# ------------------------------------
# 2) Threshold stats + plots per t
# ------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

THRESHOLDS = np.linspace(0.5, 0.9, 9)

BASE_DIR = RUN_DIR / "eval" / "threshold_analysis"
UP_DIR   = BASE_DIR / "up"
DN_DIR   = BASE_DIR / "down"
BASE_DIR.mkdir(parents=True, exist_ok=True)
UP_DIR.mkdir(parents=True, exist_ok=True)
DN_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Build threshold tables
# --------------------------
rows_up = []
rows_dn = []

for t, g in df.groupby("t_to_end_min"):
    n_total = len(g)

    for tau in THRESHOLDS:
        # ---- UP: confident UP when p_up >= tau
        sel_up = g[g["p_up"] >= tau]
        if len(sel_up) > 0:
            rows_up.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_up)),
                "coverage": float(len(sel_up) / n_total),
                "accuracy": float((sel_up["y_pred"] == sel_up["y_true"]).mean()),
                "base_rate_up": float(sel_up["y_true"].mean()),  # UP rate in selected set
            })

        # ---- DOWN: confident DOWN when p_up <= (1 - tau)
        sel_dn = g[g["p_up"] <= (1 - tau)]
        if len(sel_dn) > 0:
            # if we "act DOWN", predicted label is 0
            rows_dn.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_dn)),
                "coverage": float(len(sel_dn) / n_total),
                "accuracy": float((sel_dn["y_true"] == 0).mean()),
                "base_rate_up": float(sel_dn["y_true"].mean()),  # should be low if DOWN is correct
            })

thr_up = pd.DataFrame(rows_up)
thr_dn = pd.DataFrame(rows_dn)

# --------------------------
# Plot helper (no extra funcs)
# --------------------------
for side, thr_df, out_dir in [
    ("UP", thr_up, UP_DIR),
    ("DOWN", thr_dn, DN_DIR),
]:
    if thr_df.empty:
        continue

    for t, g in thr_df.groupby("t_to_end_min"):
        g = g.sort_values("threshold")

        plt.figure()
        ax = plt.gca()

        # Accuracy (blue)
        line1, = ax.plot(
            g["threshold"],
            g["accuracy"],
            marker="o",
            color="blue",
            label="Accuracy given predicted probability ≥ τ"
        )

        # X axis label depends on side
        if side == "UP":
            ax.set_xlabel("Probability threshold (p_up ≥ τ)")
        else:
            ax.set_xlabel("Probability threshold (p_up ≤ 1 − τ)")

        ax.set_ylabel("Accuracy given predicted probability ≥ τ")

        # Coverage (green)
        ax2 = ax.twinx()
        line2, = ax2.plot(
            g["threshold"],
            g["coverage"] * 100,
            marker="s",
            linestyle="--",
            color="green",
            label="Coverage (%)"
        )
        ax2.set_ylabel("Coverage (%)")

        # Annotate absolute counts
        for _, row in g.iterrows():
            ax.annotate(
                f"{int(row['n_samples'])}",
                (row["threshold"], row["accuracy"]),
                textcoords="offset points",
                xytext=(0, 6),
                ha="center",
                fontsize=8,
            )

        ax.grid(True)
        plt.title(f"{side}: Accuracy & coverage vs threshold (t_to_end_min={t})")

        ax.legend(handles=[line1, line2], loc="lower right")

        plt.savefig(out_dir / f"accuracy_coverage_t{t}.png", dpi=150, bbox_inches="tight")
        plt.close()

# --------------------------
# Save summary.json (both)
# --------------------------
summary = {
    "thresholds": [float(x) for x in THRESHOLDS],
    "up": thr_up.to_dict(orient="records"),
    "down": thr_dn.to_dict(orient="records"),
}

(BASE_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print("Saved:", BASE_DIR / "summary.json")
print("UP plots  :", UP_DIR)
print("DOWN plots:", DN_DIR)


/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_90418/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_90418/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_90418/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the fut

Saved: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC/eval/threshold_analysis/summary.json
UP plots  : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC/eval/threshold_analysis/up
DOWN plots: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_20251219-192004_UTC/eval/threshold_analysis/down


## Legacy Test for Check

In [44]:
# -------------------------------------------------------------
# 2) Accuracy + confusion per t_to_end_min
# -------------------------------------------------------------
# Here we go beyond global metrics and analyze performance
# as a function of "minutes to end of 15-min window".
#
# Steps:
#   1) Find where t_to_end_min lives in the feature vector
#   2) Extract t_to_end_min for each TEST sample
#      (from the *unscaled* X_test, last time step in each sequence)
#   3) Run a forward pass over test_loader to collect:
#         - predicted labels
#         - true labels
#   4) Build a DataFrame and compute:
#         - TP/FP/TN/FN per t_to_end_min
#         - accuracy per t_to_end_min
# -------------------------------------------------------------
import pandas as pd
from tqdm.auto import tqdm

# 2.1) Find the feature index for t_to_end_min
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# 2.2) Extract t_to_end_min from the UN-SCALED test data
#      Each sequence has shape (seq_len, num_features).
#      We take the LAST timestep [-1] for each sample:
#         → this corresponds to the "current" minute the model is predicting for.
t_to_end_min_values = X_test_raw[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# 2.3) Collect predictions and true labels from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(
        tqdm(test_loader, desc="Predicting (test)", leave=False)
    ):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)     # IMPORTANT: use best_model
        probs  = torch.sigmoid(logits)
        preds  = (probs >= 0.5).long()

        if batch_i == 0:
            xb = X_batch.detach().cpu().contiguous().numpy()
            lg = logits.detach().cpu().contiguous().numpy()
            pr = preds.detach().cpu().contiguous().numpy()

            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])

            print("training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)

        # IMPORTANT: extend with numpy arrays, not torch tensors
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_preds  = np.array(all_preds, dtype=int).reshape(-1)
all_labels = np.array(all_labels, dtype=int).reshape(-1)

assert (
    all_preds.shape[0] == t_to_end_min_values.shape[0]
), "Mismatch: number of test predictions != number of t_to_end_min entries"


# 2.4) Build a DataFrame for per-minute analysis
df_results = pd.DataFrame(
    {
        "t_to_end_min": t_to_end_min_values,
        "y_true": all_labels,
        "y_pred": all_preds,
    }
)

# Add confusion components per sample
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# 2.5) Aggregate stats by t_to_end_min
stats = (
    df_results.groupby("t_to_end_min")
    .agg(
        count=("correct", "count"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        tn=("tn", "sum"),
        fn=("fn", "sum"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

stats["accuracy_pct"] = stats["accuracy"] * 100

# 2.6) Print per-minute results
print("\n" + "=" * 70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("=" * 70)
print(stats.to_string(index=False))
print("=" * 70)

# 2.7) Baseline vs model
# Baseline (always predicting 1) accuracy = fraction of positives in test labels
baseline_acc = all_labels.mean()
model_acc = (all_preds == all_labels).mean()

print(f"\nBaseline Accuracy (always predict 1): {baseline_acc:.4f}")
print(f"Model Test Accuracy (from preds):      {model_acc:.4f}")

t_to_end_min is at feature index: 14
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]


Predicting (test):   0%|          | 0/29592 [00:00<?, ?it/s]

MODEL_FP: 6eae0c985503c3f6
X_FP: bafdebb3d1c993b8
LOGITS_FP: 8a4b4452a9b2f74a
PREDS_FP: 76c862828aa3b269
training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0

ACCURACY + CONFUSION METRICS PER t_to_end_min
 t_to_end_min  count    tp   fp    tn   fn  accuracy  accuracy_pct
          1.0  31564 15370  364 15459  371  0.976714     97.671398
          2.0  31564 14768 1073 14750  973  0.935179     93.517932
          3.0  31564 14360 1700 14123 1381  0.902389     90.238880
          4.0  31564 13963 2157 13666 1778  0.875333     87.533266
          5.0  31564 13588 2719 13104 2153  0.845647     84.564694
          6.0  31564 13276 3200 12623 2465  0.820523     82.052338
          7.0  31564 13054 3686 12137 2687  0.798093     79.809276
          8.0  31564 12734 4226 11597 3007  0.770847     77.084653
          9.0  31564 12394 4837 10986 3347  0.740717     74.071727
         10.0  31564 12054 5555 10268 3687  0.707198     70.719807
         11.0  31564 11851 6239

In [28]:
def hash_array(a: np.ndarray) -> str:
    return hashlib.md5(a.tobytes()).hexdigest()

print("X_test shape:", X_test_raw.shape)
print("y_test shape:", y_test.shape)
print("X_test hash:", hash_array(X_test_raw))
print("y_test hash:", hash_array(y_test))

X_test shape: (473458, 60, 15)
y_test shape: (473458,)
X_test hash: 821e37fd2e30d0dd30749865b5d2ab94
y_test hash: 7138a0d4459944e41259e786b11688ec
X_test hash: 821e37fd2e30d0dd30749865b5d2ab94
y_test hash: 7138a0d4459944e41259e786b11688ec
